# Roman Urdu captions — Colab training

**Runtime → Change runtime type → T4 GPU** before running anything.

Then **Runtime → Run all**. Every cell is idempotent and the whole thing is
resumable, so if the session drops you re-open and run all again — it picks up
from the last checkpoint on Drive rather than starting over.

Free Colab disconnects at roughly four hours, so training is stopped by a clock
(`MAX_TRAIN_MINUTES`), not by an epoch. An unfinished run that saved nothing is
worth zero; a partial adapter is worth continuing.

**Never paste a token into a cell.** A `.ipynb` stores its output, so a printed
token is committed with the file. Use the key icon in the left sidebar to add
`HF_TOKEN` as a Colab secret.

In [ ]:
# --- Settings -------------------------------------------------------------
import os
import subprocess
import time

REPO = "https://github.com/nabeeltahirdeveloper/STT-Model.git"
BRANCH = "phase0-baseline-and-spelling-spec"
HF_MODEL = "MubeenAmjad205/roman-urdu-captions"  # private
TRAIN_HOURS = 20  # audio hours to fetch (~0.62 GB each)
MAX_TRAIN_MINUTES = 150  # leaves room for download, eval and upload in 4 h

T0 = time.time()


def sh(cmd, **kw):
    print(f"$ {cmd}", flush=True)
    return subprocess.run(cmd, shell=True, check=kw.pop("check", True), **kw)


def elapsed():
    return f"{(time.time() - T0) / 60:.0f} min elapsed"

In [ ]:
# --- GPU check ------------------------------------------------------------
# Fail here rather than 40 minutes in. A CPU runtime will "work" and take days.
import torch

assert (
    torch.cuda.is_available()
), "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again."
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name} · {vram:.0f} GB VRAM")

In [ ]:
# --- Where checkpoints live ------------------------------------------------
# No Google Drive. /content is wiped when the session ends, so the private
# HuggingFace repo is the durable copy -- and it is offsite, which Drive on the
# same account is not. train pushes there every `push_every` steps and on the
# time-box, so a disconnect costs minutes rather than the run.
OUT = "/content/finetuned"
os.makedirs(OUT, exist_ok=True)
print("local:", OUT, "· durable:", HF_MODEL)

In [ ]:
# --- Repo + dependencies --------------------------------------------------
if not os.path.isdir("/content/model"):
    sh(f"git clone --branch {BRANCH} --single-branch {REPO} /content/model")
os.chdir("/content/model")
sh("git pull --ff-only", check=False)

sh("pip -q install uv")
# bitsandbytes and accelerate are declared in the `train` extra, not installed
# separately: `uv pip install` writes into an environment that the next
# `uv run` re-syncs and prunes, so the package vanishes before it is used.
sh("uv sync --extra train --extra cuda")

# Fail here, not 40 minutes in.
sh('uv run python -c "import bitsandbytes; print(f"bitsandbytes {bitsandbytes.__version__}")"')
print(elapsed())

In [ ]:
# --- Auth: from Colab secrets, never from a cell --------------------------
from google.colab import userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    from huggingface_hub import whoami

    print("hugging face:", whoami()["name"])
except Exception as e:
    print("No HF_TOKEN secret — training will run, upload will be skipped.")
    print("Add it with the key icon in the left sidebar.", type(e).__name__)

In [ ]:
# --- Data -----------------------------------------------------------------
# Transcripts are tiny; audio is the slow part. Both download inside Google's
# network, far faster than a home connection, and both are resumable.
sh(
    "uv run python -m scripts.download_transcripts --corpus-transcripts",
    check=False,
)

# data/labels/ is gitignored, so the clone has no labels -- they are generated
# rather than shipped. Dictionary lookup makes that ~3 seconds for 29,749
# utterances, which is why they are not worth versioning (ADR-014).
sh("uv run python -m scripts.build_labels --split US-CS")

sh(f"uv run python -m scripts.select_training_subset --hours {TRAIN_HOURS}")
sh(
    "uv run python -m scripts.fetch_training_audio --manifest data/labels/train-subset.jsonl",
    check=False,
)
print(elapsed())

In [ ]:
# --- Train: FULL fine-tune, not LoRA --------------------------------------
# LoRA on these labels scored CER 64.1% against 34.9% for the stock model plus
# our romanizer -- worse than not training. Full fine-tuning is the recipe
# ADR-003 specifies; 8-bit Adam plus gradient checkpointing is what makes
# ~780M trainable parameters fit in a 16 GB T4.
#
# Verify on 60 clips before spending the session. This check takes under a
# minute and would have caught every mistake made getting here.
sh(f"uv run python -m src.training.finetune --limit 60 --out {OUT}-smoke")

resume = f"--resume {OUT}" if os.path.exists(f"{OUT}/config.json") else ""
remaining = max(15, MAX_TRAIN_MINUTES - (time.time() - T0) / 60)
sh(
    f"uv run python -m src.training.finetune --out {OUT} {resume} "
    f"--max-minutes {remaining:.0f} --hf-repo {HF_MODEL} --push-every 300"
)
print(elapsed())

In [ ]:
# --- The gate: CER, not script mix ----------------------------------------
# This session's hard lesson. Script mix went 5.7% -> 100% Latin across three
# runs and was reported as success; CER had meanwhile gone 34.9% -> 64.1%. The
# model got twice as wrong while looking twice as good.
sh(
    "uv run python -m scripts.download_transcripts --corpus-transcripts",
    check=False,
)

sh(f"uv run python -m src.eval.score --pred {OUT}/predictions.txt", check=False)
print("\nCompare against 34.9% -- stock 0.6B plus our romanizer on the same clips.")
print("Above it means training did not help, whatever else looks good.")

In [ ]:
# --- Already uploaded ------------------------------------------------------
# train pushes to the private HF repo during the run, so there is nothing to do
# here unless the run was interrupted before its first push.
print("model ->", HF_MODEL, "(private)")
print(elapsed())

## If the session dies

Re-open this notebook and **Run all**. The repo is already cloned, the audio is
already downloaded, and training resumes from the Drive checkpoint. Nothing is
repeated that does not need to be.

## Reading the result

The only number that matters is **CER against the eval set**. Compare it with
stock 0.6B plus our romanizer, which scored **34.9%** on the same clips.

- **Below 34.9%** — training helped. That is the first real evidence it does.
- **Above** — it did not, whatever the script mix says. Lower the learning rate
  (`--learning-rate 2e-5`) and check the loss curve is falling rather than
  rising before spending another session on it.

Do not compare against the 27.9% baseline: that is the 1.7B model on the full
set, and ADR-016 explains why the comparison is not like for like.